# Расчёт нормы калорий и КБЖУ (болезни + стадии жизни)

Считает суточную норму калорий и распределение **Б/Ж/У** с учётом:
- пола, возраста, веса, роста;
- уровня физической активности;
- цели — поддержание / снижение / набор веса;
- **нозологической группы** (здоровый, диабет 2 типа, ожирение, ХБП, ССЗ);
- **стадии жизни** (спортсмен, пожилой, беременная, кормящая).

**БЖУ зависит от обоих параметров.** При конфликте белка между болезнью и стадией жизни приоритет у ХБП (защита почек). Рискованные комбинации отмечаются предупреждениями.

> ⚠️ Расчёт ориентировочный, не заменяет консультацию врача/диетолога.

**Как пользоваться:** заполни ячейку «Данные человека» ниже и запусти все ячейки (`Shift+Enter`).

## 1. Импорты

In [1]:
import sys, os
# Чтобы `import diet` работал из папки notebooks/.
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd

from diet import (UserProfile, calculate, CONDITIONS, LIFE_STAGES,
                  ACTIVITY_LEVELS, GOALS)

pd.set_option('display.unicode.east_asian_width', True)
print('Группы здоровья (condition):', list(CONDITIONS))
print('Стадии жизни (life_stage):', list(LIFE_STAGES))
print('Уровни активности:', list(ACTIVITY_LEVELS))
print('Цели:', list(GOALS))

Группы здоровья (condition): ['healthy', 'diabetes_t2', 'obesity', 'ckd', 'cvd']
Стадии жизни (life_stage): ['default', 'athlete_endurance', 'athlete_strength', 'older_adult', 'pregnant', 'lactating']
Уровни активности: ['sedentary', 'light', 'moderate', 'high', 'very_high']
Цели: ['maintain', 'lose', 'gain']


## 2. Данные человека

**condition:** `healthy` / `diabetes_t2` / `obesity` / `ckd` / `cvd`   
**life_stage:** `default` / `athlete_endurance` / `athlete_strength` / `older_adult` / `pregnant` / `lactating`  
**sex:** `male` / `female`  
**activity:** `sedentary` / `light` / `moderate` / `high` / `very_high`  
**goal:** `maintain` / `lose` / `gain`  
**formula:** `who` / `mifflin`

In [2]:
# ↓↓↓ ЗАПОЛНИ ПОД СЕБЯ ↓↓↓
profile_data = {
    "sex": "male",        # male / female
    "age": 25,              # полных лет
    "weight": 78,           # кг
    "height": 180,          # см
    "activity": "light",   # sedentary / light / moderate / high / very_high
    "goal": "maintain",    # maintain / lose / gain
}
condition  = "healthy"      # healthy / diabetes_t2 / obesity / ckd / cvd
life_stage = "default"      # default / athlete_endurance / athlete_strength / older_adult / pregnant / lactating
formula    = "who"          # who / mifflin

profile = UserProfile(**profile_data)

Рекомпозиция

## 3. Расчёт

In [3]:
result = calculate(profile, formula=formula, condition=condition, life_stage=life_stage)

## 4. Сравнение формул BMR

In [4]:
pd.DataFrame(result.comparison()).T

,ВОЗ (Скофилд),Миффлин-Сан Жеор
"BMR, ккал",1870,1785
"Расход за день (TDEE), ккал",2571,2454


## 5. Итог

In [5]:
df = pd.DataFrame.from_dict(result.summary(), orient='index', columns=['Значение'])
df

,Значение
Группа здоровья,Здоровый взрослый
Стадия жизни,Обычный взрослый
Формула BMR,who
"BMR по ВОЗ, ккал",1870
"BMR по Миффлину, ккал",1785
"BMR используемый, ккал",1870
"Расход за день (TDEE), ккал",2571
Уровень активности,Небольшая (1-3 тренировки/нед)
Цель,Поддержание веса
"Целевые калории, ккал/день",2571


In [6]:
total_macro_kcal = result.protein_kcal + result.fat_kcal + result.carbs_kcal

print(f"Группа здоровья: {result.condition_label}")
print(f"Стадия жизни: {result.life_stage_label}")
print(f"Формула: {result.formula} · Цель: {result.goal_label} · Активность: {result.activity_label}")
print(f"Целевая калорийность: {round(result.target_kcal)} ккал/день")
print()
print(f"Белки:     {round(result.protein_g):>4} г  ({result.protein_kcal:>4} ккал, {result.protein_kcal/total_macro_kcal*100:>4.1f}%)")
print(f"Жиры:      {round(result.fat_g):>4} г  ({result.fat_kcal:>4} ккал, {result.fat_kcal/total_macro_kcal*100:>4.1f}%)")
print(f"Углеводы:  {round(result.carbs_g):>4} г  ({result.carbs_kcal:>4} ккал, {result.carbs_kcal/total_macro_kcal*100:>4.1f}%)")

Группа здоровья: Здоровый взрослый
Стадия жизни: Обычный взрослый
Формула: who · Цель: Поддержание веса · Активность: Небольшая (1-3 тренировки/нед)
Целевая калорийность: 2571 ккал/день

Белки:       96 г  ( 386 ккал, 15.0%)
Жиры:        86 г  ( 771 ккал, 30.0%)
Углеводы:   353 г  (1414 ккал, 55.0%)


## 6. Предупреждения и рекомендации

Если выбрана рискованная комбинация (ХБП + спортсмен, беременность + похудение/диабет), она отобразится здесь.

In [7]:
if result.warnings:
    print('⚠️  ПРЕДУПРЕЖДЕНИЯ:')
    for i, w in enumerate(result.warnings, 1):
        print(f'  {i}. {w}')
else:
    print('Предупреждений нет — комбинация безопасная.')
print()
if result.notes:
    print('Рекомендации по выбранной группе/стадии:')
    print(result.notes)

Предупреждений нет — комбинация безопасная.



## 7. Сравнение по стадиям жизни (бонус)

Как меняется БЖУ при одном профиле и группе здоровья в зависимости от стадии жизни.

In [8]:
rows = []
for ls in LIFE_STAGES:
    r = calculate(profile, formula=formula, condition=condition, life_stage=ls)
    rows.append({
        "Стадия жизни": r.life_stage_label,
        "Калории": round(r.target_kcal),
        "Белки, г": round(r.protein_g),
        "Жиры, г": round(r.fat_g),
        "Углеводы, г": round(r.carbs_g),
        "Предупр.": '⚠' if r.warnings else '',
    })
pd.DataFrame(rows)

,Стадия жизни,Калории,"Белки, г","Жиры, г","Углеводы, г",Предупр.
0,Обычный взрослый,2571,96,86,353,
1,Спортсмен на выносливость,2571,101,86,348,
2,Спортсмен-силовик / набор массы,2571,140,86,309,
3,Пожилой (60+),2571,86,86,364,
4,Беременность (2-3 триместр),2911,134,97,375,
5,Кормление грудью (лактация),3071,140,102,397,
